# Create metric views

Creates (or replaces) one Unity Catalog metric view per YAML file in `gold/metric_views/`. The view name is the file name.
The YAML files use `${catalog}`, `${silver_schema}` and `${gold_schema}` placeholders, filled in from the parameters below,
so the same files work for every bundle target. Runs as the last task of the pipelines job, after gold.

In [0]:
import os
import re
from string import Template

dbutils.widgets.text("catalog", "mg_aws_serverless_workspace_catalog", "Catalog")
dbutils.widgets.text("silver_schema", "sdp_meta_silver", "Silver schema")
dbutils.widgets.text("gold_schema", "sdp_meta_gold", "Gold schema (metric views land here)")

PARAMS = {k: dbutils.widgets.get(k) for k in ("catalog", "silver_schema", "gold_schema")}
for k, v in PARAMS.items():
    # Values are spliced into DDL and YAML, so allow plain identifiers only.
    assert re.fullmatch(r"[A-Za-z0-9_]+", v), f"invalid {k}: {v!r}"

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
REPO_ROOT = "/Workspace" + os.path.dirname(os.path.dirname(notebook_path)).removeprefix("/Workspace")
MV_DIR = f"{REPO_ROOT}/gold/metric_views"
print(PARAMS, MV_DIR, sep="\n")

In [0]:
import yaml

created = []
for file_name in sorted(os.listdir(MV_DIR)):
    if not file_name.endswith((".yml", ".yaml")):
        continue
    view = os.path.splitext(file_name)[0]
    with open(f"{MV_DIR}/{file_name}") as f:
        yaml_body = Template(f.read()).substitute(PARAMS)  # fails on any unknown placeholder
    yaml.safe_load(yaml_body)  # clear local error for YAML syntax mistakes (e.g. unquoted ': ')
    fqn = f"`{PARAMS['catalog']}`.`{PARAMS['gold_schema']}`.`{view}`"
    spark.sql(f"CREATE OR REPLACE VIEW {fqn}\nWITH METRICS\nLANGUAGE YAML\nAS $$\n{yaml_body}\n$$")
    created.append(fqn)
    print("created", fqn)
assert created, f"no metric view YAML files found in {MV_DIR}"

In [0]:
for fqn in created:
    view = fqn.split(".")[-1].strip("`")
    with open(f"{MV_DIR}/{view}.yml") as f:
        spec = yaml.safe_load(f)
    measures = ", ".join(f"MEASURE(`{m['name']}`) AS `{m['name']}`" for m in spec["measures"])
    row = spark.sql(f"SELECT {measures} FROM {fqn}").first().asDict()
    print(fqn, row, sep="\n  ")
    assert all(v is not None for v in row.values()), f"{fqn}: a measure returned NULL"